In [ ]:
# ===== E0 — img1280 cache তৈরি (CPU, ~৪৫-৬০ মিনিট) =====
# কাজ: ছবিগুলো 1280px-এ আবার cache করা। এটাই বাকি সব কাজের ভিত্তি।
#
# কেন — মূল সমস্যাটা এখানে:
#   আসল ছবি ~2066x1209 PNG। plan_a-র cache সেটাকে 384px-এ নামায় (384x225)।
#   ওই cache বানানো হয়েছিল CLIP/DINOv2-র জন্য, যাদের মাত্র 224px লাগে।
#   কিন্তু পরে আসা প্রতিটা শক্তিশালী model ওই thumbnail-টাই উত্তরাধিকারে পেয়েছে:
#
#     Qwen3-VL-Embedding-2B : 384x225 -> ~১১২ visual token  (আসল ছবিতে হতো ~৩,১৮২)
#     VLM captioning        : thumbnail দেখে caption লিখছে
#     SigLIP-so400m-384     : 384x225 কে টেনে 384x384 — উল্লম্ব ~৫৯% ফাঁপা
#     OCR                   : গড় ৫৪.৬ অক্ষর, ১২.৩% একদম ফাঁকা
#
#   Qwen3-VL dynamic-resolution model — যত pixel, তত token। পুরো repo-তে
#   min_pixels/max_pixels কোথাও সেট করা নেই, মানে cache-ই একমাত্র বাধা, model নয়।
#
# 1280 কেন: 1280x749 ≈ 959k pixel, যা max_pixels ~1M (≈১২৮০ token)-এর সাথে
# ঠিক মেলে — VRAM-এ যতটা কুলোয় ততটাই।
#
# Accelerator: None (CPU)।  Internet: OFF।
# Input: astroclimb (competition) + essentials
#
# ⏱️ Resume: যে ছবি হয়ে গেছে সেটা আবার হয় না। session কাটলে Output-কে Dataset
#    বানিয়ে পরের run-এ Input দাও — বাকিটুকু শেষ করবে।
import os, glob, time, gc, hashlib, base64, shutil
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

COMP = '/kaggle/input/competitions/astroclimb'
OUT, INP = '/kaggle/working', '/kaggle/input'
IMGD = f'{OUT}/img1280'
os.makedirs(IMGD, exist_ok=True)
T0 = time.time()
def tlog(*a): print(f'[{(time.time()-T0)/60:6.1f} min]', *a, flush=True)
def find(n, isdir=False):
    for r in (INP, OUT):
        for h in glob.glob(f'{r}/**/{n}', recursive=True):
            if os.path.isdir(h) == isdir: return h
    return None

assert os.path.isdir(COMP), f'{COMP} নেই — competition dataset attach করো'

# ---- আগের run-এর ছবি থাকলে আগে OUT-এ এনে নিই, যাতে শেষে একটাই সম্পূর্ণ ফোল্ডার হয় ----
prev = None
for d in glob.glob(f'{INP}/**/img1280', recursive=True):
    if os.path.isdir(d): prev = d; break
if prev:
    have = set(os.listdir(IMGD))
    n = 0
    for f in os.listdir(prev):
        if f not in have:
            shutil.copy2(f'{prev}/{f}', f'{IMGD}/{f}'); n += 1
    tlog(f'আগের run থেকে {n}টা ছবি আনা হলো ({prev})')
DONE = {f[:-4] for f in os.listdir(IMGD) if f.endswith('.jpg')}
tlog(f'আগে থেকে আছে: {len(DONE)}')

In [ ]:
# ===== E0 — CELL 1 : cache তৈরি =====
# hash plan_a.ipynb CELL 8-এর হুবহু এক — একটুও আলাদা হলে পরের প্রতিটা join
# চুপচাপ ফাঁকা দেবে, কোনো error ছাড়াই।
PNG  = 'iVBORw0KGgo'
SIDE = 1280          # ছিল 384
QUAL = 92            # ছিল 90
def obj_hash(s): return hashlib.md5(s.encode('utf-8','replace')).hexdigest()

BUDGET = 10.5 * 3600
seen, fails, made = set(DONE), [], 0

def handle(args):
    h, s = args
    try:
        img = Image.open(BytesIO(base64.b64decode(s)))
        img.draft('RGB', (SIDE, SIDE))        # PNG-তে no-op, plan_a-র সাথে মিল রাখতে
        img = img.convert('RGB')
        img.thumbnail((SIDE, SIDE), Image.LANCZOS)   # শুধু ছোট করে, aspect রাখে
        img.save(f'{IMGD}/{h}.jpg', 'JPEG', quality=QUAL)
        return h, img.size, None
    except Exception as e:
        return h, None, repr(e)[:90]

sizes, stop = [], False
with ThreadPoolExecutor(4) as ex:            # PIL decode/encode GIL ছাড়ে, তাই thread-ই যথেষ্ট
    for path in (f'{COMP}/train.csv', f'{COMP}/test.csv'):
        if stop: break
        tlog('streaming', os.path.basename(path))
        for ch in pd.read_csv(path, chunksize=200):
            batch = []
            for col in ('obj_1','obj_2'):
                for s in ch[col].values:
                    if not isinstance(s, str) or not s.startswith(PNG): continue
                    h = obj_hash(s)
                    if h in seen: continue
                    seen.add(h); batch.append((h, s))
            for h, sz, err in ex.map(handle, batch):
                if err: fails.append((h, err))
                else:
                    made += 1
                    if len(sizes) < 300: sizes.append(sz)
            del ch, batch; gc.collect()
            if made and made % 1000 < 400 and time.time()-T0 > 30:
                tlog(f'  বানানো {made} | মোট ফাইল {len(os.listdir(IMGD))} | ব্যর্থ {len(fails)}')
            if time.time()-T0 > BUDGET:
                tlog('⏱️ সময়সীমা — Output→Dataset বানিয়ে আবার চালাও'); stop = True; break

tlog(f'শেষ — নতুন {made} | ফোল্ডারে মোট {len(os.listdir(IMGD))} | ব্যর্থ {len(fails)}')
if fails: print('ব্যর্থ নমুনা:', fails[:5])
if sizes:
    w = np.array([s[0] for s in sizes]); h_ = np.array([s[1] for s in sizes])
    print(f'\nআকার (প্রথম {len(sizes)}টা): প্রস্থ গড় {w.mean():.0f} সর্বোচ্চ {w.max()} | '
          f'উচ্চতা গড় {h_.mean():.0f} সর্বোচ্চ {h_.max()}')
    print(f'pixel গড় {(w*h_).mean()/1e6:.2f} Mpx   (পুরনো 384 cache-এ ছিল ~0.086 Mpx)')

In [ ]:
# ===== E0 — CELL 2 : 🚨 hash যাচাই (এটাই একমাত্র নীরব failure) =====
IHP = find('img_hashes.parquet')
assert IHP, 'img_hashes.parquet নেই — essentials attach করো'
IH = pd.read_parquet(IHP).hash.astype(str).values
have = {f[:-4] for f in os.listdir(IMGD) if f.endswith('.jpg')}

miss_a, miss_b = set(IH) - have, have - set(IH)
print(f'img_hashes.parquet : {len(IH)}')
print(f'আমার বানানো ছবি     : {len(have)}')
print(f'ওখানে আছে আমার নেই  : {len(miss_a)}')
print(f'আমার আছে ওখানে নেই  : {len(miss_b)}')

if miss_a and not miss_b:
    print(f'\n⚠️ {len(miss_a)}টা ছবি এখনো বাকি — Output→Dataset বানিয়ে notebook আবার চালাও।')
elif miss_b:
    raise AssertionError(
        'hash মিলছে না — plan_a-র সাথে আলাদা হয়ে গেছে। এগিয়ো না, নাহলে পরের '
        f'প্রতিটা join ফাঁকা দেবে। নমুনা: {list(miss_b)[:3]}')
else:
    print('\n✅ ১৪,৫৯৮টা hash হুবহু মিলেছে — সব পরের notebook নিরাপদে join করতে পারবে')

# আকারের তুলনা — সত্যিই বড় হয়েছে কিনা
old = find('img384', isdir=True)
if old and have:
    import random
    s = random.Random(0).sample(sorted(have), min(5, len(have)))
    print(f'\n{"hash":34s} {"পুরনো 384":>14s} {"নতুন 1280":>14s}  pixel বৃদ্ধি')
    for h in s:
        try:
            a = Image.open(f'{old}/{h}.jpg').size
            b = Image.open(f'{IMGD}/{h}.jpg').size
            print(f'{h:34s} {str(a):>14s} {str(b):>14s}  {(b[0]*b[1])/(a[0]*a[1]):.1f}x')
        except Exception: pass

print('\n👉 Save Version → Output কে Dataset বানাও (নাম: img1280)।')
print('   এরপর E1 (OCR) আর E3 (Qwen hi-res embedding) দুটোই এটা Input হিসেবে নেবে।')
tlog('done')